In [2]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))


In [12]:

import matplotlib.pyplot as plt

from scripts.main import load_graph_and_communities, run_configured_sicp_intermediates
from core.removal import remove_nodes, ept_bridge_in_strength_removal, make_adaptive_removal

hg,  communities = load_graph_and_communities("Algebra", tau=2, verbose=True, source=False)

hedge_size = { hedge : len(nodes) for hedge, nodes in hg.hyperedges.items()}
ones = { hedge for hedge in hedge_size.keys() if hedge_size[hedge] == 5}

print(ones)
raise

BETA = 0.02
N = len(hg.nodes)
K = int(N * 0.7)

adaptive_ept_bis = make_adaptive_removal(ept_bridge_in_strength_removal, batch=1)

rem = remove_nodes(hg, adaptive_ept_bis(hg, communities, BETA, K=K))

print(K)
print(N - len(rem.nodes))



Loaded dataset 'Algebra' (edge-labeled) with 1268 hyperedges and 10 communities
  communities sizes: min=2, max=88, avg=42.300
{4, 7, 1034, 524, 525, 1038, 21, 539, 1051, 545, 35, 1060, 39, 1072, 1077, 1084, 575, 1092, 1097, 79, 594, 595, 87, 1115, 93, 620, 1139, 1140, 632, 1145, 638, 640, 641, 130, 645, 145, 1171, 662, 155, 1185, 163, 675, 1195, 1198, 691, 185, 187, 700, 1218, 708, 709, 1233, 214, 728, 218, 1244, 225, 227, 1258, 1264, 1265, 761, 253, 256, 261, 778, 268, 285, 801, 302, 817, 306, 311, 320, 839, 333, 859, 348, 349, 350, 353, 869, 358, 361, 375, 892, 895, 385, 388, 398, 912, 427, 435, 436, 953, 956, 449, 452, 455, 456, 972, 462, 975, 976, 475, 988, 990, 1000, 1003, 1005, 1022}


RuntimeError: No active exception to reraise

In [ ]:
import statistics

import matplotlib.pyplot as plt

from scripts.main import load_graph_and_communities, run_configured_sicp_intermediates


hg,  communities = load_graph_and_communities("Geometry", tau=2, verbose=True, source=False)


BETA = 0.2
T = 25
RUNS = 10
RNG_SEED = 175
SEED_ITERATIONS = 200



edge_nodes = set().union(*hg.hyperedges.values()) if hg.hyperedges else set()
extra_nodes = hg.nodes - edge_nodes

res_old = run_configured_sicp_intermediates(
    hg,
    seed_iterations=SEED_ITERATIONS,
    beta=BETA,
    T=T,
    runs=RUNS,
    rng_seed=RNG_SEED,
)


N = len(hg.nodes)                      # total number of nodes in the simulation
timesteps = list(range(T + 1))
prevalence = [x / N for x in res_old]  # convert mean infected count -> mean prevalence

plt.figure(figsize=(8, 5))
plt.plot(timesteps, prevalence, marker="o")
plt.xlabel("Timestep")
plt.ylabel("Prevalence")
plt.title("SICP prevalence over time")
plt.ylim(0, 1)
plt.xlim(0, T)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Single experiment: contact-high-school, no nodes removed, immediate intermediate plot
import statistics

import matplotlib.pyplot as plt

from scripts.main import load_graph_and_communities, run_configured_sicp
from core.removal import remove_nodes, remove_nodes_greedily, degree_based_removal, make_adaptive_removal
from core.data_loader import get_graph_nodes

BETA = 0.2
T = 25
RUNS = 10
RNG_SEED = 175
SEED_ITERATIONS = 200

graph, communities = load_graph_and_communities("Algebra", tau=2, verbose=True, source=False)

N = len(get_graph_nodes(graph))
K = int(0.6 * float(N))

g_old = remove_nodes_greedily(degree_based_removal, [graph], K, 1)
g_new = remove_nodes(graph, make_adaptive_removal(degree_based_removal, 1)(graph, K=K))
normal_removed_graph = remove_nodes(graph, degree_based_removal(graph, K=K))

print(f"N: {N}, K: {K}, rem: {N - K}")
print(len(get_graph_nodes(g_old)))
print(len(get_graph_nodes(g_new)))

res_old = run_configured_sicp(
    g_old,
    seed_iterations=SEED_ITERATIONS,
    beta=BETA,
    T=T,
    runs=RUNS,
    rng_seed=RNG_SEED,
)

res_new = run_configured_sicp(
    g_new,
    seed_iterations=SEED_ITERATIONS,
    beta=BETA,
    T=T,
    runs=RUNS,
    rng_seed=RNG_SEED,
)

prevalence_non_greedy = run_configured_sicp(
    normal_removed_graph,
    seed_iterations=SEED_ITERATIONS,
    beta=BETA,
    T=T,
    runs=RUNS,
    rng_seed=RNG_SEED,
)

print(statistics.mean(res_old.values() ))
print(statistics.mean(res_new.values()))
print(statistics.mean(prevalence_non_greedy.values() ))


